# 20 HistGradientBoosting Optuna Prep for J_D

## Import

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import joblib

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [ ]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [ ]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

## Hilfsvariablen

In [ ]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [ ]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

## Preproc nach J_D

In [ ]:
targ_enc = TargetEncoder(
    cv=5,
    shuffle=True,
    random_state=RANDOM_STATE,
    smooth=50.0
)

In [ ]:
te_train = pd.DataFrame(
    targ_enc.fit_transform(x_train[cat_cols], y_train),
    columns=cat_cols,
    index=x_train.index
)

te_val = pd.DataFrame(
    targ_enc.transform(x_val[cat_cols]),
    columns=cat_cols,
    index=x_val.index
)

te_test = pd.DataFrame(
    targ_enc.transform(x_test[cat_cols]),
    columns=cat_cols,
    index=x_test.index
)

In [ ]:
x_train_hgb = x_train.copy()
x_val_hgb = x_val.copy()
x_test_hgb = x_test.copy()

x_train_hgb[cat_cols] = te_train
x_val_hgb[cat_cols] = te_val
x_test_hgb[cat_cols] = te_test

In [ ]:
x_train_hgb_no_calc = x_train_hgb[feat_cols_no_calc]
x_val_hgb_no_calc = x_val_hgb[feat_cols_no_calc]
x_test_hgb_no_calc = x_test_hgb[feat_cols_no_calc]

In [ ]:
pd.Series({
    "train": [len(x_train), len(x_train_hgb), len(x_train_hgb_no_calc)],
    "val": [len(x_val), len(x_val_hgb), len(x_val_hgb_no_calc)],
    "test": [len(x_test), len(x_test_hgb), len(x_test_hgb_no_calc)]
})

## Optuna Hilfe

## Suchräume

|Parameter|Bereich|
|---|---|
|learning_rate|0.003-0.3 (log)|
|min_samples_leaf|5-40000 (log)|
|l2_regularization|1e-8 - 1e5 (log)|
|max_features|0.05-0.35|
|max_bins|104-255|
|interaction_cst|None / pairwise / no_interactions|
|validation_fraction|0.01-0.1|
|n_iter_no_change|5-40|
|tol|1e-10 - 1e-5 (log)|

Für händische Versuche entweder:
|Parameter|Bereich|
|---|---|
|max_leaf_nodes|2-63|
|max_depth|1-6|

In [ ]:
fixed_params_leaf = {
    "random_state": RANDOM_STATE,
    "verbose": 0,
    "loss": "log_loss",
    "class_weight": None,
    "categorical_features": None,
    "scoring": "loss",
    "max_iter": 5000,
    "early_stopping": True,
    "max_depth": None
}

In [ ]:
fixed_params_depth = {
    "random_state": RANDOM_STATE,
    "verbose": 0,
    "loss": "log_loss",
    "class_weight": None,
    "categorical_features": None,
    "scoring": "loss",
    "max_iter": 5000,
    "early_stopping": True,
    "max_leaf_nodes": None
}

In [ ]:
def suchraum_params_leaf(trial):
    """Suchraum definition, leaf"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40000, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 1e5, log=True),
        "max_features": trial.suggest_float("max_features", 0.05, 0.35),
        "max_bins": trial.suggest_int("max_bins", 104, 255),
        "interaction_cst": trial.suggest_categorical("interaction_cst", [None, "pairwise", "no_interactions"]),
        "validation_fraction": trial.suggest_float("validation_fraction", 0.01, 0.1),
        "n_iter_no_change": trial.suggest_int("n_iter_no_change", 5, 40),
        "tol": trial.suggest_float("tol", 1e-10, 1e-5, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 63)
    }

    return params

In [ ]:
def suchraum_params_depth(trial):
    """Suchraum definition, depth"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40000, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 1e5, log=True),
        "max_features": trial.suggest_float("max_features", 0.05, 0.35),
        "max_bins": trial.suggest_int("max_bins", 104, 255),
        "interaction_cst": trial.suggest_categorical("interaction_cst", [None, "pairwise", "no_interactions"]),
        "validation_fraction": trial.suggest_float("validation_fraction", 0.01, 0.1),
        "n_iter_no_change": trial.suggest_int("n_iter_no_change", 5, 40),
        "tol": trial.suggest_float("tol", 1e-10, 1e-5, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 6)
    }

    return params

## Leaf mit Calc

In [ ]:
def objective_leaf_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_leaf,
        **suchraum_params_leaf(trial)
    )

    modell.fit(x_train_hgb, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb)[:, 1])

In [ ]:
study_leaf_calc = optuna.create_study(
    study_name = "HGB_leaf_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_leaf_calc.optimize(
    objective_leaf_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_leaf_calc.best_value,
    "best gini": 2 * study_leaf_calc.best_value - 1,
    "trials": len(study_leaf_calc.trials),
    "pruned": len([t for t in study_leaf_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_leaf_calc.trials if t.state.name == "FAIL"]),
    "best params": study_leaf_calc.best_params
})

In [ ]:
study_leaf_calc.best_params

## Leaf ohne Calc

In [ ]:
def objective_leaf_no_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_leaf,
        **suchraum_params_leaf(trial)
    )

    modell.fit(x_train_hgb_no_calc, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb_no_calc)[:, 1])

In [ ]:
study_leaf_no_calc = optuna.create_study(
    study_name = "HGB_leaf_no_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_leaf_no_calc.optimize(
    objective_leaf_no_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_leaf_no_calc.best_value,
    "best gini": 2 * study_leaf_no_calc.best_value - 1,
    "trials": len(study_leaf_no_calc.trials),
    "pruned": len([t for t in study_leaf_no_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_leaf_no_calc.trials if t.state.name == "FAIL"]),
    "best params": study_leaf_no_calc.best_params
})

In [ ]:
study_leaf_no_calc.best_params

## Depth mit Calc

In [ ]:
def objective_depth_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_depth,
        **suchraum_params_depth(trial)
    )

    modell.fit(x_train_hgb, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb)[:, 1])

In [ ]:
study_depth_calc = optuna.create_study(
    study_name = "HGB_depth_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_depth_calc.optimize(
    objective_depth_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_depth_calc.best_value,
    "best gini": 2 * study_depth_calc.best_value - 1,
    "trials": len(study_depth_calc.trials),
    "pruned": len([t for t in study_depth_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_depth_calc.trials if t.state.name == "FAIL"]),
    "best params": study_depth_calc.best_params
})

In [ ]:
study_depth_calc.best_params

## Depth ohne Calc

In [ ]:
def objective_depth_no_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_depth,
        **suchraum_params_depth(trial)
    )

    modell.fit(x_train_hgb_no_calc, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb_no_calc)[:, 1])

In [ ]:
study_depth_no_calc = optuna.create_study(
    study_name = "HGB_depth_no_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_depth_no_calc.optimize(
    objective_depth_no_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_depth_no_calc.best_value,
    "best gini": 2 * study_depth_no_calc.best_value - 1,
    "trials": len(study_depth_no_calc.trials),
    "pruned": len([t for t in study_depth_no_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_depth_no_calc.trials if t.state.name == "FAIL"]),
    "best params": study_depth_no_calc.best_params
})

In [ ]:
study_depth_no_calc.best_params

## Load Study

In [ ]:
study_leaf_calc_loaded = optuna.load_study(
    study_name = "HGB_leaf_calc",
    storage = "sqlite:///HGB_opti.db"
)


In [ ]:
study_leaf_no_calc_loaded = optuna.load_study(
    study_name = "HGB_leaf_no_calc",
    storage = "sqlite:///HGB_opti.db"
)

In [ ]:
study_depth_calc_loaded = optuna.load_study(
    study_name = "HGB_depth_calc",
    storage = "sqlite:///HGB_opti.db"
)


In [ ]:
study_depth_no_calc_loaded = optuna.load_study(
    study_name = "HGB_depth_no_calc",
    storage = "sqlite:///HGB_opti.db"
)

## Best Params again

In [ ]:
study_leaf_calc_loaded.best_params

In [ ]:
study_leaf_no_calc_loaded.best_params

In [ ]:
study_depth_calc_loaded.best_params

In [ ]:
study_depth_no_calc_loaded.best_params

## 100% Train

In [ ]:
results = []
train_times = {}

In [ ]:
name = "L_hgb_opt_01"

L_hgb_opt_01 = HistGradientBoostingClassifier(
    **fixed_params_leaf,
    **study_leaf_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_01.fit(
    x_train_hgb, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_01, x_train_hgb, y_train, x_val_hgb, y_val, train_times[name], best_iter=L_hgb_opt_01.n_iter_)
)

In [ ]:
name = "L_hgb_opt_02"

L_hgb_opt_02 = HistGradientBoostingClassifier(
    **fixed_params_leaf,
    **study_leaf_no_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_02.fit(
    x_train_hgb_no_calc, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_02, x_train_hgb_no_calc, y_train, x_val_hgb_no_calc, y_val, train_times[name], best_iter=L_hgb_opt_02.n_iter_)
)

In [ ]:
name = "L_hgb_opt_03"

L_hgb_opt_03 = HistGradientBoostingClassifier(
    **fixed_params_depth,
    **study_depth_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_03.fit(
    x_train_hgb, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_03, x_train_hgb, y_train, x_val_hgb, y_val, train_times[name], best_iter=L_hgb_opt_03.n_iter_)
)

In [ ]:
name = "L_hgb_opt_04"

L_hgb_opt_04 = HistGradientBoostingClassifier(
    **fixed_params_depth,
    **study_depth_no_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_04.fit(
    x_train_hgb_no_calc, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_04, x_train_hgb_no_calc, y_train, x_val_hgb_no_calc, y_val, train_times[name], best_iter=L_hgb_opt_04.n_iter_)
)

In [ ]:
pd.DataFrame(results)

## Save Models

In [ ]:
Modelle = [
    (L_hgb_opt_01, "L_hgb_opt_01"),
    (L_hgb_opt_02, "L_hgb_opt_02"),
    (L_hgb_opt_03, "L_hgb_opt_03"),
    (L_hgb_opt_04, "L_hgb_opt_04")
]

for modell, name in Modelle:
    joblib.dump(modell, f"{name}.joblib")

joblib.dump(targ_enc, "L_hgb_targ_enc.joblib")

## Notizen
- Ich habe bis jetzt noch nichts zum pruning für HGB gefunden: Mines Verständnisses nach ist die Pruner implementation wie XGBPruner eine Schnittstelle durch die während des trainings gepruned werden kann. Ich setze den Median Pruner trotzdem, aber mMn wird dieser einfach keinen Effekt haben.
- J_D hatte erwähnt entweder max_depth oder max_leaf_nodes zu setzen, ich behandle es wie plain und ordered als händische versuchsreihe
- ich muss über joblib speichern
- da jd etwas anders als ich target encoded speichere ich den target encoder von hier auch mal

## Ressourcen [Abrufdatum: 19.08.2026]:
- https://www.kaggle.com/code/adrienriaux/histgradientboosting-with-optuna
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
- https://sklearngeneticopt.rodrigo-arenas.com/versions/latest/tutorials/tune-gradient-boosting
- https://www.cosmiclearn.com/scikitlearn/joblib.php
- https://www.geeksforgeeks.org/machine-learning/saving-a-machine-learning-model/
- https://www.simplified.guide/scikit-learn/model-persist-joblib